In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df = pd.read_csv("data.csv", )
pd.set_option('display.max_columns', None)

# Set display option to show float values in standard notation
pd.set_option('display.float_format', lambda x: '%.2f' % x)

display(df.head())

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Id,Customer Segment,Customer State,Customer Zipcode,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,Order_Date,Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Name,Product Price,Product Status,Shipping_Date,Shipping Mode,Post_Order_Action,Transportation_Cost,Lead_Time,Supplier_ID
0,DEBIT,3,4,91.25,314.64,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,20755,Consumer,PR,725.00,Fitness,18.25,-66.04,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.11,0.04,180517,327.75,0.29,1,327.75,314.64,91.25,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,NaN,23.64,6,sup001
1,TRANSFER,5,4,-249.09,311.36,Late delivery,1,Sporting Goods,Caguas,Puerto Rico,19492,Consumer,PR,725.00,Fitness,18.28,-66.04,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.39,0.05,179254,327.75,-0.80,1,327.75,311.36,-249.09,South Asia,Rajastán,PENDING,NaN,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,NaN,18.31,10,sup004
2,CASH,4,4,-247.78,309.72,Shipping on time,0,Sporting Goods,San Jose,EE. UU.,19491,Consumer,CA,95125.00,Fitness,37.29,-121.88,Pacific Asia,Bikaner,India,19491,2018-01-13 12:06:00,75938,1360,18.03,0.06,179253,327.75,-0.80,1,327.75,309.72,-247.78,South Asia,Rajastán,CLOSED,NaN,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,NaN,21.52,8,sup004
3,DEBIT,3,4,22.86,304.81,Advance shipping,0,Sporting Goods,Los Angeles,EE. UU.,19490,Home Office,CA,90027.00,Fitness,34.13,-118.29,Pacific Asia,Townsville,Australia,19490,2018-01-13 11:45:00,75937,1360,22.94,0.07,179252,327.75,0.08,1,327.75,304.81,22.86,Oceania,Queensland,COMPLETE,NaN,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,NaN,21.39,6,sup002
4,PAYMENT,2,4,134.21,298.25,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,19489,Corporate,PR,725.00,Fitness,18.25,-66.04,Pacific Asia,Townsville,Australia,19489,2018-01-13 11:24:00,75936,1360,29.50,0.09,179251,327.75,0.45,1,327.75,298.25,134.21,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,NaN,26.17,4,sup001


## **Data Preparation and Analysis**

In this section, we will perform the following tasks:

1. **Convert Order Date to Datetime**:
   Convert the 'Order_Date' column to datetime format to facilitate time-based calculations.

2. **Calculate Time Period**:
   Compute the time period between the earliest and latest order dates in days.

3. **Group Data by Product**:
   Aggregate data by 'Product Card Id' to compute total order quantities, average product price, average lead time, and total sales.

4. **Calculate Ordering Cost**:
   Determine the ordering cost by applying a fixed multiplier to the product price.

5. **Calculate Standard Deviation of Daily Demand**:
   Compute the standard deviation of daily order quantities for each product.

6. **Merge Standard Deviation with Inventory Data**:
   Merge the standard deviation data with the aggregated inventory DataFrame to include variability in demand.


In [2]:
# Convert order date to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# Calculate the time period between the greatest and lowest date
time_period = df['Order_Date'].max() - df['Order_Date'].min()
# Convert time_period to number of days
days_period = time_period.days


# Group by product 

inventory_df = df.groupby(['Product Card Id']).agg({
    'Order Item Quantity': 'sum',
    'Product Price':'mean',
    'Lead_Time':'mean',
    'Sales':'sum'
    # Add other columns you want to aggregate here
})

#Calculating Ordering Cost
inventory_df['Ordering_Cost']=inventory_df['Product Price']*0.6
# Rename columns for clarity
inventory_df.rename(columns={'Product Card Id': 'Product_Id', 'Order Item Quantity':'Demand_Rate'}, inplace=True)

# Calculate standard deviation of daily demand for each product
std_dev_demand = df.groupby('Product Card Id')['Order Item Quantity'].std().reset_index()
std_dev_demand.rename(columns={'Order Item Quantity': 'Std_Dev_Demand'}, inplace=True)

# Merge with std_dev_demand to include the standard deviation of demand
inventory_df = pd.merge(inventory_df, std_dev_demand, on='Product Card Id')


# Display the new DataFrame
display(inventory_df.head())


,Product Card Id,Demand_Rate,Product Price,Lead_Time,Sales,Ordering_Cost,Std_Dev_Demand
0,19,64,124.99,7.77,7999.36,74.99,0.00
1,24,231,79.99,7.08,18477.69,47.99,1.31
2,35,65,159.99,6.69,10399.35,95.99,0.00
3,37,781,34.99,6.89,27327.19,20.99,1.41
4,44,939,59.99,7.00,56330.61,35.99,1.46


### Calculating EOQ (Economic Order Quantity) and JIT (Just-In-Time)

**Economic Order Quantity (EOQ)**: Determines the optimal order quantity to minimize total inventory costs.

**Just-In-Time (JIT) Inventory Management**: Focuses on reducing inventory levels and associated costs by synchronizing inventory with actual demand.

- **Reorder Point (ROP)**: The inventory level at which a new order should be placed to prevent stockouts. It ensures that stock is replenished before it runs out.
- **Lead Time**: The time required from placing an order to receiving it. Effective management of lead time ensures timely replenishment and aligns inventory levels with demand.
- **Daily Demand**: The average demand per day, calculated from the total demand rate. It helps in setting appropriate reorder points and determining how much inventory is needed each day.

These components work together in JIT to ensure inventory is available just when needed, minimizing excess stock and reducing carrying costs.





In [3]:

# Handle Null Values
inventory_df = inventory_df.fillna(inventory_df.mean())

# Define parameters for EOQ calculation (example values)
inventory_df['Holding_Cost'] = inventory_df['Product Price']*0.07  # Holding cost per unit per year
inventory_df['Lead_Time'] = np.random.randint(3, 8, size=len(inventory_df))

# Calculate Reorder Point (ROP)
inventory_df['Daily_Demand'] = inventory_df['Demand_Rate'] / days_period 
inventory_df['Reorder_Point'] = inventory_df['Daily_Demand'] * inventory_df['Lead_Time']

# Calculate EOQ
inventory_df['EOQ'] = np.sqrt((2 * inventory_df['Demand_Rate'] * inventory_df['Ordering_Cost']) / inventory_df['Holding_Cost'])
pd.set_option('display.max_rows', None)


display(inventory_df.head())


,Product Card Id,Demand_Rate,Product Price,Lead_Time,Sales,Ordering_Cost,Std_Dev_Demand,Holding_Cost,Daily_Demand,Reorder_Point,EOQ
0,19,64,124.99,7,7999.36,74.99,0.00,8.75,0.06,0.40,33.12
1,24,231,79.99,7,18477.69,47.99,1.31,5.60,0.21,1.44,62.93
2,35,65,159.99,4,10399.35,95.99,0.00,11.20,0.06,0.23,33.38
3,37,781,34.99,5,27327.19,20.99,1.41,2.45,0.69,3.47,115.71
4,44,939,59.99,6,56330.61,35.99,1.46,4.20,0.83,5.00,126.87


## **ABC Analysis**

**ABC Analysis** is an inventory management technique used to categorize inventory items into three categories (A, B, and C) based on their importance and value. The goal is to prioritize inventory management efforts on the most critical items.

- **Category A**: Items with the highest value and typically the lowest quantity. These items are crucial for operations and require strict inventory control and frequent review.
- **Category B**: Items with moderate value and quantity. These items are important but not as critical as Category A items, requiring regular but less frequent review.
- **Category C**: Items with the lowest value and highest quantity. These items are less critical and require minimal management effort and infrequent review.

**Purpose**: Helps allocate resources effectively, improve inventory turnover, and reduce holding costs by focusing management efforts on high-impact items.


In [4]:
# Step 2: Rank items by total value
df_sorted = inventory_df.sort_values(by='Sales', ascending=False).reset_index(drop=True)

# Step 3: Calculate cumulative percentage
df_sorted['Cumulative_Value'] = df_sorted['Sales'].cumsum()
df_sorted['Cumulative_Percentage'] = df_sorted['Cumulative_Value'] / df_sorted['Sales'].sum() * 100

# Step 4: Categorize items into A, B, C
def categorize(row):
    if row['Cumulative_Percentage'] <= 80:
        return 'A'
    elif row['Cumulative_Percentage'] <= 95:
        return 'B'
    else:
        return 'C'

df_sorted['Category'] = df_sorted.apply(categorize, axis=1)

random_rows = df_sorted.sample(n=5, random_state=40) 
# Display the randomly selected rows
display(random_rows)


,Product Card Id,Demand_Rate,Product Price,Lead_Time,Sales,Ordering_Cost,Std_Dev_Demand,Holding_Cost,Daily_Demand,Reorder_Point,EOQ,Cumulative_Value,Cumulative_Percentage,Category
81,1059,40,349.99,7,13999.60,209.99,0.00,24.50,0.04,0.25,26.19,36426648.95,99.03,C
51,273,795,27.99,6,22252.05,16.79,1.46,1.96,0.71,4.24,116.74,35871151.53,97.52,C
6,403,22246,129.99,4,2891757.66,77.99,0.00,9.10,19.76,79.03,617.54,28276258.35,76.87,A
45,282,848,31.99,7,27127.52,19.19,1.47,2.24,0.75,5.27,120.57,35731275.21,97.14,C
54,567,859,25.00,3,21475.00,15.00,1.46,1.75,0.76,2.29,121.35,35936558.95,97.69,C


## Calculating Safety Stock

**Safety Stock** is extra inventory kept to prevent stockouts due to variations in demand or delays in supply. It ensures you have enough stock to cover unexpected changes.

**Components**:
- **Z-Score (Z)**: This value represents the desired service level. For example, a Z-score of 1.645 is used for a 95% service level, meaning there's a 95% chance that inventory will not be depleted.
- **Standard Deviation of Demand During Lead Time**: This measures how much demand fluctuates during the lead time. It is calculated by taking the standard deviation of daily demand and adjusting it for the length of the lead time.

**Purpose**: To calculate how much extra inventory is needed to avoid running out of stock, even with uncertainties in demand or delivery times.


In [5]:

# Desired service level (e.g., 95% service level)
service_level = 0.95

# Calculate the Z-score for the desired service level using the inverse of the CDF for a standard normal distribution
Z = stats.norm.ppf(service_level)

# Calculate standard deviation of demand during lead time
inventory_df['Std_Dev_Demand_Lead_Time'] = inventory_df['Std_Dev_Demand'] * np.sqrt(inventory_df['Lead_Time'])

# Calculate Safety Stock
inventory_df['Safety_Stock'] = Z * inventory_df['Std_Dev_Demand_Lead_Time']

display(inventory_df.head())



,Product Card Id,Demand_Rate,Product Price,Lead_Time,Sales,Ordering_Cost,Std_Dev_Demand,Holding_Cost,Daily_Demand,Reorder_Point,EOQ,Std_Dev_Demand_Lead_Time,Safety_Stock
0,19,64,124.99,7,7999.36,74.99,0.00,8.75,0.06,0.40,33.12,0.00,0.00
1,24,231,79.99,7,18477.69,47.99,1.31,5.60,0.21,1.44,62.93,3.47,5.72
2,35,65,159.99,4,10399.35,95.99,0.00,11.20,0.06,0.23,33.38,0.00,0.00
3,37,781,34.99,5,27327.19,20.99,1.41,2.45,0.69,3.47,115.71,3.15,5.18
4,44,939,59.99,6,56330.61,35.99,1.46,4.20,0.83,5.00,126.87,3.57,5.87
